# MPI


In [ ]:
import os
import sys

# Add the '../sources' directory to the Python path using source_dir,
# and add its parent directory as well.
source_dir = os.path.abspath(os.path.join("..", "sources"))
parent_dir = os.path.abspath(os.path.join(source_dir, os.pardir))

for directory in [source_dir, parent_dir]:
    if directory not in sys.path:
        sys.path.insert(0, directory)

print(f"Included '{source_dir}' and its parent directory '{parent_dir}' in sys.path.")

In [ ]:
## Define the materials
from sources.photonic_crystal_maker import Material
from sources.photonic_crystal_maker import Lattice, Geometry, PhotonicCrystal
from sources.mpb_configurator import MPBSchemeConfigurator
import meep as mp
from sources.photonic_crystal_maker import Lattice, Geometry, PhotonicCrystal
from sources.mpb_configurator import *


n_InP = 3.1

eps = round(n_InP**2,2)
print("Epsilon of InP: ", eps)
InP = Material(epsilon = eps)
air = Material(epsilon = 1)
## Simulation Name: 
name = "test_mpi"

## Define the lattice

lattice = Lattice(type="TXY")
centers = lattice.get_centers()
k_points = lattice.get_get_k_points_around_Gamma(distance = 0.5)

## Define the geometry

radius1 = Geometry.make_script_param(r1=0.2)
radius2 = Geometry.make_script_param(r2=0.3)
bulk = Geometry(geom_type=mp.Block, params={"center": mp.Vector3(0,0,0), "size": mp.Vector3(1e20, 1e20, 1e20), "material": InP})
hole1 = Geometry(geom_type=mp.Cylinder, params={"center": centers[0], "radius": radius1, "height": 1e20, "material": air})
hole2 = Geometry(geom_type=mp.Cylinder, params={"center": centers[1], "radius": radius2, "height": 1e20, "material": air})
photonic_crystal = PhotonicCrystal(atoms=[bulk, hole1, hole2], lattice=lattice)

## Define the simulation

configuration_options_opt = dict(
    resolution=32,
    num_bands=8,
    k_points=[mp.Vector3(0, 0, 0)],
    k_point_interpolation_factor=None, 
)
mpb_config_opt = MPBSchemeConfigurator(photonic_crystal, ["te", "tm"], **configuration_options_opt)
script_opt = mpb_config_opt.get_scheme_config(join_newline=True)

configuration_options_single = dict(
    resolution=64,
    num_bands=8,
    k_points=k_points["k_points_values"],
    k_point_interpolation_factor=20, 
)

mpb_config_single = MPBSchemeConfigurator(photonic_crystal, ["te", "tm"], **configuration_options_single)
script_single = mpb_config_single.get_scheme_config(join_newline=True)


In [ ]:
from sources.mpi_differential_evolution import MPIDiffEvoSimulation
from sources.simulation_handler import Simulation
from sources.simulation_handler import SimulationViewer

# Create your Simulation instance.
sim_single = Simulation(simulation_name=name,
                 script=script_single,
                 log_level = Simulation.WARNING)

sim_single.run_hpc(mpb_command_line_params={"r1":0.10, "r2":0.339})

df =sim_single.load_frequency_data("te")
freqs = sim_single.get_frequencies_by_band(df, "tm")
print(freqs)
cost = abs(freqs[4] - freqs[2])


viewer = SimulationViewer(sim_single)
viewer.plot_epsilon(periods=3)
fig = viewer.plot_band_diagram("te", colors = "red", k_points_labels=k_points["k_points_labels"], k_points_values=k_points["k_points_values"])
fig = viewer.plot_band_diagram("tm", fig=fig, colors = "blue", k_points_labels=k_points["k_points_labels"], k_points_values=k_points["k_points_values"])  


In [ ]:
# Instantiate the MPI optimization wrapper.
optimizer = MPIDiffEvoSimulation(simulation_name = "lsf_test", scheme_script= script_opt, maxiter=200,  param_names=["r1", "r2"], param_bounds=[(0, 0.35), (0.1, 0.5)], polarization="tm")
# Submit the LSF job.
submission_info = optimizer.submit_lsf_job(nprocs=24, walltime="00:30", queue="fotonano", span_option="block", span_value=4)
print("Submission info:", submission_info)

In [ ]:
def plot_optimization_points(self, 
                     log_file_path=None, 
                     use_logscale=False, 
                     levels=50, 
                     points_only=False,
                     plot_inverse_cost=False,
                     custom_title=None):
    """
    Reads lines from the .log file, extracting arbitrary parameter names and 
    their values, along with 'cost'. Produces a 2D heat map (or scatter plot) 
    of 'cost' (or 1/cost) vs. two selected parameters.

    We assume each line has the form:
        paramA: <float>, paramB: <float>, ..., cost: <float>
    and specifically that exactly 2 parameters + 1 'cost' are present
    in the lines we want to plot. Lines that do not meet these criteria
    are skipped.

    Parameters
    ----------
    log_file_path : str, optional
        Path to the log file. If None, uses self.log_file.

    use_logscale : bool, optional
        If True, the color scale is displayed in log scale (requires all
        cost values to be > 0, or if plot_inverse_cost=True, then 1/cost
        must be > 0). Defaults to False.

    levels : int or None, optional
        - If an integer (e.g., 50), uses tricontourf with that many discrete
          contour levels.
        - If None, uses tripcolor with Gouraud shading (continuous).
        - Ignored if points_only=True.

    points_only : bool, optional
        If True, skip triangulation/contours entirely and just plot
        the raw points, colored by cost. Defaults to False.

    plot_inverse_cost : bool, optional
        If True, plot 1/cost instead of cost. This can be useful if you
        want to highlight small cost values as large color-mapped values.
        Make sure your cost is never zero. Defaults to False.

    custom_title : str, optional
        If provided, this string overrides the default title. Defaults to None.
    """
    import re
    import matplotlib.pyplot as plt
    import os
    from matplotlib.colors import LogNorm

    if log_file_path is None:
        log_file_path = self.log_file

    # We'll store the param data in separate arrays for plotting:
    x_vals = []
    y_vals = []
    cost_vals = []

    # Track the parameter names for labeling
    param_x_name = None
    param_y_name = None

    if not os.path.isfile(log_file_path):
        print(f"Log file not found: {log_file_path}")
        return

    with open(log_file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip empty lines

            # Regex to find all "paramName: value" pairs
            pattern = r"(\w[\w\d_]*)\s*:\s*([\d.+\-eE]+)"
            matches = re.findall(pattern, line)
            if not matches:
                # no param-value pairs found
                continue

            # Convert to a dictionary: { paramName: floatValue }
            param_dict = {}
            for (pname, pval_str) in matches:
                try:
                    val = float(pval_str)
                except ValueError:
                    continue
                param_dict[pname] = val

            # We expect one to be 'cost'
            cost = param_dict.pop('cost', None)
            if cost is None:
                continue  # no cost found

            # If we want 1/cost, handle that now
            if plot_inverse_cost:
                if cost == 0:
                    # If cost=0 is ever in the log, skip this line to avoid divide-by-zero
                    continue
                cost = 1.0 / cost

            # We need exactly 2 other parameters
            if len(param_dict) != 2:
                continue  # skip lines that don't match exactly 2 params

            # Sort param names so that we always pick them in a stable order
            sorted_params = sorted(param_dict.keys())
            p1, p2 = sorted_params[0], sorted_params[1]

            # Lock in param names once we see the first valid line
            if param_x_name is None and param_y_name is None:
                param_x_name, param_y_name = p1, p2
            else:
                # If we encounter a line with different param names, skip
                if {p1, p2} != {param_x_name, param_y_name}:
                    continue

            x_val = param_dict[param_x_name]
            y_val = param_dict[param_y_name]

            x_vals.append(x_val)
            y_vals.append(y_val)
            cost_vals.append(cost)

    # Check if we have any valid data
    if not x_vals:
        print("No valid lines with exactly two parameters + cost found.")
        return

    # If logscale is requested but we have non-positive data, revert to linear
    if use_logscale:
        min_cost = min(cost_vals)
        if min_cost <= 0:
            print("Cannot use log scale because min cost <= 0. Switching to linear scale.")
            use_logscale = False

    # Prepare norm for matplotlib
    norm = LogNorm(vmin=min(cost_vals), vmax=max(cost_vals)) if use_logscale else None

    # Prepare some strings for the default title
    scale_title = " (Log Scale)" if use_logscale else " (Linear Scale)"
    extra_title = "1/Cost" if plot_inverse_cost else "Cost"

    # Function to decide final plot title if custom_title is None
    def make_title(prefix):
        if custom_title is not None:
            return custom_title
        else:
            return f"{prefix}{scale_title} ({extra_title})"

    # If user wants only points (scatter)
    if points_only:
        plt.figure(figsize=(7, 6))
        scatter = plt.scatter(x_vals, y_vals, c=cost_vals, cmap="viridis", norm=norm)
        plt.colorbar(scatter, label=extra_title)
        plt.xlabel(param_x_name)
        plt.ylabel(param_y_name)
        plt.title(make_title("Parameter Space Scatter"))
        plt.tight_layout()
        plt.show()
        return

    # Otherwise, attempt contour or tripcolor
    try:
        import matplotlib.tri as mtri
        triang = mtri.Triangulation(x_vals, y_vals)

        plt.figure(figsize=(7, 6))

        if levels is None:
            # Continuous shading with tripcolor
            pc = plt.tripcolor(
                triang,
                cost_vals,
                shading="gouraud",
                cmap="viridis",
                norm=norm
            )
            plt.colorbar(pc, label=extra_title)
            plot_desc = "Tripcolor (Gouraud Shading)"
        else:
            # Discrete contours with tricontourf
            cntr = plt.tricontourf(
                triang,
                cost_vals,
                levels=levels,
                cmap="viridis",
                norm=norm
            )
            plt.colorbar(cntr, label=extra_title)
            plot_desc = f"Tricontourf (levels={levels})"

        plt.xlabel(param_x_name)
        plt.ylabel(param_y_name)
        plt.title(make_title(f"Parameter Space Heatmap\n{plot_desc}"))
        plt.tight_layout()
        plt.show()

    except Exception as e:
        # Fallback to a scatter plot
        print("Falling back to scatter plot due to:", e)
        plt.figure(figsize=(7, 6))
        scatter = plt.scatter(x_vals, y_vals, c=cost_vals, cmap="viridis", norm=norm)
        plt.colorbar(scatter, label=extra_title)
        plt.xlabel(param_x_name)
        plt.ylabel(param_y_name)
        plt.title(make_title("Parameter Space Scatter"))
        plt.tight_layout()
        plt.show()


In [ ]:
plot_optimization_points(optimizer, log_file_path="lsf_test/lsf_test.log", use_logscale=True, levels = 100, points_only=True, plot_inverse_cost=True, custom_title="Optimization Points")  

# MPI MAPPER

In [ ]:
import os
import sys

# Add the '../sources' directory to the Python path using source_dir,
# and add its parent directory as well.
source_dir = os.path.abspath(os.path.join("..", "sources"))
parent_dir = os.path.abspath(os.path.join(source_dir, os.pardir))

for directory in [source_dir, parent_dir]:
    if directory not in sys.path:
        sys.path.insert(0, directory)

print(f"Included '{source_dir}' and its parent directory '{parent_dir}' in sys.path.")

In [ ]:
## Define the materials
from sources.photonic_crystal_maker import Material
from sources.photonic_crystal_maker import Lattice, Geometry, PhotonicCrystal
from sources.mpb_configurator import MPBSchemeConfigurator
import meep as mp
from sources.mpb_configurator import *


n_InP = 3.1

eps = round(n_InP**2,2)
print("Epsilon of InP: ", eps)
InP = Material(epsilon = eps)
air = Material(epsilon = 1)
## Simulation Name: 
name = "test_mpi"

## Define the lattice
lattice = Lattice(type="TXY")
centers = lattice.get_centers()
k_points = lattice.get_get_k_points_around_Gamma(distance = 0.5)

## Define the geometry
radius1 = Geometry.make_script_param(r1=0.2)
radius2 = Geometry.make_script_param(r2=0.3)
bulk = Geometry(geom_type=mp.Block, params={"center": mp.Vector3(0,0,0), "size": mp.Vector3(1e20, 1e20, 1e20), "material": InP})
hole1 = Geometry(geom_type=mp.Cylinder, params={"center": centers[0], "radius": radius1, "height": 1e20, "material": air})
hole2 = Geometry(geom_type=mp.Cylinder, params={"center": centers[1], "radius": radius2, "height": 1e20, "material": air})
photonic_crystal = PhotonicCrystal(atoms=[bulk, hole1, hole2], lattice=lattice)

## Define the simulation
configuration_options_opt = dict(
    resolution=32,
    num_bands=8,
    k_points=[mp.Vector3(0, 0, 0)],
    k_point_interpolation_factor=None, 
)
mpb_config_opt = MPBSchemeConfigurator(photonic_crystal, ["te", "tm"], **configuration_options_opt)
script_opt = mpb_config_opt.get_scheme_config(join_newline=True)



configuration_options_single = dict(
    resolution=64,
    num_bands=8,
    k_points=k_points["k_points_values"],
    k_point_interpolation_factor=20, 
)
mpb_config_single = MPBSchemeConfigurator(photonic_crystal, ["te", "tm"], **configuration_options_single)
script_single = mpb_config_single.get_scheme_config(join_newline=True)


In [ ]:
import os
from sources.mpi_gap_mapper import MPIGapMapper  # Adjust the import based on your file location

# Set up simulation details
simulation_name = "test_mapper"



# Define mapping parameters
param_names = ["r1", "r2"]
grid_bounds = [(0.05, 0.15), (0.2, 0.4)]  # boundaries for each parameter
resolution = (50, 50)                   # 50 grid points for each parameter
polarization = "tm"
band_indices = (2, 4)                   # gap = |freq[4] - freq[2]|

# Instantiate the gap mapper
mapper = MPIGapMapper(simulation_name=simulation_name,
                      scheme_script=script_opt,
                      param_names=param_names,
                      grid_bounds=grid_bounds,
                      resolution=resolution,
                      polarization=polarization,
                      band_indices=band_indices)

# Run the mapping and plot the heatmap.
# (For LSF, you would call mapper.submit_lsf_job(...) to submit and wait.)
mapper.submit_lsf_job(nprocs=32, walltime="00:30", queue="fotonano", span_option="block", span_value=4)


In [ ]:
    def plot_mapping_from_log(self, log_file_path=None, use_logscale=False, levels=50, 
                              points_only=False, custom_title=None, plot_inverse_gap=False):
        """
        Reads lines from the mapping log file, extracting parameter names and values,
        and plots a heatmap (or scatter plot) of gap versus the two parameters.
        
        Each valid log line is expected to contain key-value pairs in the form:
            <param1>: <value>, <param2>: <value>, gap: <value>
        The method extracts the parameter names dynamically from the first valid line.
        
        Parameters
        ----------
        log_file_path : str, optional
            Path to the log file. If None, uses self.mapping_log.
        use_logscale : bool, optional
            If True, applies a logarithmic color normalization.
        levels : int or None, optional
            If an integer, uses tricontourf with that many levels;
            if None, uses tripcolor with Gouraud shading.
            Ignored if points_only is True.
        points_only : bool, optional
            If True, produces a scatter plot instead of a contour plot.
        custom_title : str, optional
            Custom title for the plot.
        plot_inverse_gap : bool, optional
            If True, plots 1/gap instead of gap.
        """
        from matplotlib.colors import LogNorm
        
        if log_file_path is None:
            log_file_path = self.mapping_log

        # Lists to store parameter values and gap.
        param1_vals = []
        param2_vals = []
        gap_vals = []
        # We will also determine the parameter names dynamically.
        x_label, y_label = None, None
        
        if not os.path.exists(log_file_path):
            print(f"Log file {log_file_path} not found.")
            return
        
        with open(log_file_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                # Extract all key-value pairs from the line.
                pairs = re.findall(r"(\w[\w\d_]*)\s*:\s*([\d\.\-eE]+)", line)
                if not pairs:
                    continue
                # Build a dictionary of pairs.
                data = {}
                for key, val in pairs:
                    try:
                        data[key] = float(val)
                    except ValueError:
                        continue
                # Check that 'gap' is present and exactly two other parameters exist.
                if "gap" not in data or len(data) - 1 != 2:
                    continue
                gap_val = data.pop("gap")
                # Get parameter names in the order they appeared.
                keys = list(data.keys())
                if x_label is None and y_label is None:
                    x_label, y_label = keys[0], keys[1]
                # If the keys don't match the first valid line, skip this line.
                if set(keys) != {x_label, y_label}:
                    continue
                param1_vals.append(data[x_label])
                param2_vals.append(data[y_label])
                gap_vals.append(gap_val)
        
        if not param1_vals:
            print("No valid data found in the log file.")
            return

        # If the option is set to plot 1/gap, update gap values and label.
        if plot_inverse_gap:
            # Avoid division by zero.
            gap_vals = [1.0/g if g != 0 else 0 for g in gap_vals]
            gap_label = "1/Gap"
        else:
            gap_label = "Gap"
        
        norm = LogNorm(vmin=min(gap_vals), vmax=max(gap_vals)) if use_logscale else None
        
        if points_only:
            plt.figure(figsize=(8,6))
            sc = plt.scatter(param1_vals, param2_vals, c=gap_vals, cmap="viridis", norm=norm)
            plt.colorbar(sc, label=gap_label)
            plt.xlabel(x_label)
            plt.ylabel(y_label)
            title = custom_title if custom_title else "Mapping Data (Scatter)"
            plt.title(title)
            plt.tight_layout()
            plt.show()
        else:
            try:
                import matplotlib.tri as mtri
                triang = mtri.Triangulation(param1_vals, param2_vals)
                plt.figure(figsize=(8,6))
                if levels is None:
                    pc = plt.tripcolor(triang, gap_vals, shading="gouraud", cmap="viridis", norm=norm)
                    plt.colorbar(pc, label=gap_label)
                    plot_desc = "Tripcolor (Gouraud Shading)"
                else:
                    cntr = plt.tricontourf(triang, gap_vals, levels=levels, cmap="viridis", norm=norm)
                    plt.colorbar(cntr, label=gap_label)
                    plot_desc = f"Tricontourf (levels={levels})"
                plt.xlabel(x_label)
                plt.ylabel(y_label)
                title = custom_title if custom_title else f"Mapping Data ({plot_desc})"
                plt.title(title)
                plt.tight_layout()
                plt.show()
            except Exception as e:
                print("Contour plot failed; falling back to scatter plot:", e)
                plt.figure(figsize=(8,6))
                sc = plt.scatter(param1_vals, param2_vals, c=gap_vals, cmap="viridis", norm=norm)
                plt.colorbar(sc, label=gap_label)
                plt.xlabel(x_label)
                plt.ylabel(y_label)
                title = custom_title if custom_title else "Mapping Data (Scatter)"
                plt.title(title)
                plt.tight_layout()
                plt.show()


In [ ]:
log_file = "test_mapper/test_mapper_mapping.log"
plot_mapping_from_log(mapper, log_file_path=log_file, use_logscale=False, levels=None, points_only=True, custom_title="Mapping Heatmap", plot_inverse_gap=True)